In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-2: Preferences, reward models and RLHF

## Where this session fits

> **RLLM-1's verifier needs a program that can check a completion. For style, tone and helpfulness no such program exists, so we need another source of reward: people's comparisons.**

- **The question:** how does a reward come from human comparisons, and how is it used?
- **Position: session 2 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-1 named three sources of a reward: a verifier, a reward model, a judge model
    - Intro session C1 (Reinforcement learning on a language model) glossed reward models in one slide
    - this session is the full story
    - the next session, RLLM-3, replaces the human labeler by a model
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**You are here**

<img src="images/series_map_2.png" width=90%>

**By the end you can**

1. say what preference data is and why comparisons can be easier to collect than scores
2. state the Bradley-Terry model, and compute a probability and a loss from two rewards
3. say how a reward model is built and trained
4. read the three stages of RLHF, and say why stage 3 keeps the policy near a reference model
5. explain how one reward per completion reaches the per-token machinery of Part B
6. explain reward over-optimization, and why it is RLHF's real limit


## Preference data

**Why this section?** If no program can score a completion, people can still say which of two is better. The unit of data is a comparison.

> **A preference example is a triple: a prompt, a preferred completion, and a non-preferred completion.**

- a **reward** is a number whose size means something
- a **preference** says "A is better than B", with no size
- a **preference example** is $(\x, \y^+, \y^-)$: prompt, preferred completion, non-preferred completion
- example: prompt "How do I change a tire?"
    - $\y^+$: an accurate step-by-step completion
    - $\y^-$: a correct but terse completion, without jack placement or tightening order
    - both are acceptable; the data records an ordering, not correctness
- **where the completions come from:** usually the SFT model, sampled twice or more per prompt (as in InstructGPT, our reading)
    - so the data is fixed and off-policy: it will not follow the policy as RL moves it

**Why compare, and what it costs**

- asking for a number is hard: is this completion a 7 or an 8?
    - raters may use the scale differently (our reading)
    - choosing the better of two is a simpler judgment (our reading)
- the price: a comparison carries less information than a number, and raters still disagree on some pairs
- comparisons win when numbers are unreliable
- when a program can check the answer, RLLM-1's verifier is better
- the data can be used in two ways
    - imitate the preferred completion (supervised fine-tuning, SFT)
    - **learn a reward from the contrast**, and optimize it with RL (this session)


## An example preference dataset (aside)

> **UltraFeedback (binarized) is a dataset of (prompt, chosen, rejected) triples, labeled by a model rather than by people.**

- [UltraFeedback (binarized)](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized): columns *prompt*, *chosen*, *rejected*
- built from prompts, with completions from several different language models
    - a strong language model (GPT-4) rates each completion, and the ratings rank them
    - **binarized:** the best-rated completion is *chosen*
    - one of the others, at random, is *rejected*
- labels from a model are **AI feedback**: the next session, RLLM-3, studies that choice
- **off-policy:** the completions come from other models, not from the model being trained


## From comparisons to a reward: the Bradley-Terry model

**Why this section?** Reinforcement learning needs a reward per completion. We fit one so that its differences explain the comparisons.

> **Bradley-Terry: the probability that $\y^+$ is preferred is the sigmoid of the difference of two rewards. Fit the reward model by maximum likelihood.**

- the **reward model** $\rew_\psi(\x, \y)$ is a network with parameters $\psi$ (not $\phi$, which Intro session B3, actor-critic, used for the critic)
    - it gives one reward to a whole completion, read at the last token, which has seen the whole completion
- the **Bradley-Terry model**: $P(\y^+ \succ \y^-) = \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
    - $\sigma$ is the sigmoid, and $\succ$ reads "is preferred to"
    - why a sigmoid of a difference: it is 0.5 at a tie, rises toward 1 as the gap grows, and sees only the gap
- the loss, averaged over the data: $-\log \sigma\bigl(\rew_\psi(\x, \y^+) - \rew_\psi(\x, \y^-)\bigr)$
- **the network scores each completion alone; only the training compares two**

**The classifier view, by numbers**

- fitting $\psi$ is ordinary binary classification: "which of the two completions won?"
    - the reward difference is the logit (the network's raw output before the sigmoid)
    - the loss is the usual binary cross-entropy, as in logistic regression
- example: rewards $2.0$ for $\y^+$ and $0.5$ for $\y^-$, a gap of $1.5$
    - $P(\y^+ \succ \y^-) = \sigma(1.5) \approx 0.82$ and the loss is $-\log 0.82 \approx 0.20$
    - in the reverse order, $\sigma(-1.5) \approx 0.18$ and the loss is about $1.70$


## Bradley-Terry: what the loss does

<img src="images/rllm2_bradley_terry.png" width=75%>

- only **differences** matter: adding a constant to every reward of a prompt changes nothing
    - so the reward has no absolute zero, and a constant shift cancels in any advantage
    - in stage 3 what counts is the size of reward gaps against the KL penalty, which $\beta$ trades off
- the loss keeps falling as the gap widens, so a reward model can overfit its noisy labels (our reading)


## Building a reward model

**Why this section?** The reward model is itself a trained network, with choices that matter.

> **A reward model is usually a language model with a scalar output, trained on comparisons.**

- **architecture:** a language model (often the SFT model) whose next-token output is replaced by a **scalar head**: one output number
- **data:** several completions per prompt, ranked by raters
    - $K$ ranked completions give $K(K-1)/2$ pairs (4 give 6)
    - those pairs share completions, so they are correlated: more prompts help more than more pairs (our reading)
- **training:** the Bradley-Terry loss on those pairs
- **what it does not do:** it does not know the truth, only what raters tended to prefer
- weaknesses we expect (our reading)
    - it may favor long completions or a confident tone
    - it is reliable only near the completions it was trained on


# RLHF: the three stages

**Why this section?** The route from comparisons back to the Part B machinery. One design choice needs a reason first.

> **RLHF has three stages: SFT, a reward model, and RL against that reward model. Stage 3 also penalizes drift from the SFT model.**

<img src="images/rllm2_rlhf.png" width=95%>

- **RLHF** is reinforcement learning from human feedback
- **SFT** is supervised fine-tuning on **demonstrations**: completions written by people
- SFT supplies the starting policy, the reference model and, usually, the reward model's initial parameters
- stage 3 needs only prompts: no labeled completions

**Why a reference model and a KL term?**

- the reward model is trusted only near the completions it was trained on (evidence: the over-optimization slide below)
- so stage 3 keeps the policy near a frozen copy, the **reference model** $\pi_{\text{ref}}$ (here the SFT model)
- the **KL divergence** measures how far two distributions are apart. The penalty at each token is
    - $-\beta \log \dfrac{\pi_\theta(\y_{\tt} \mid \x, \y_{<\tt})}{\pi_{\text{ref}}(\y_{\tt} \mid \x, \y_{<\tt})}$ (Intro session C1)
    - for one sampled token it is a **sample estimate** of the KL
    - the log-ratio can have either sign
    - over tokens the policy samples it averages to the KL (RLLM-5 returns to it)
- $\beta$ is the KL coefficient: large stays close, small chases the reward


## Stage 3: where one reward per completion goes

**Why this section?** Part B credited every token. The reward model gives one reward for the whole completion.

> **The reward model's reward arrives once, at the last token. The KL penalty arrives at every token. The Part B machinery spreads credit.**

<img src="images/rllm2_token_rewards.png" width=72%>

- (schematic: invented sizes)
- PPO: the critic and GAE (B3, actor-critic) carry credit from the last token back to earlier ones
- GRPO: the completion's advantage (C2, GRPO) applies to all its tokens, with the KL as a loss term
- nothing new in the optimizer: the reward model is one more source of reward
- the cost: models in memory
    - three: the policy, the reference and the reward model
    - PPO adds the old policy and a critic: five


## Reward over-optimization

**Why this section?** The reward model is a proxy for what people want. Pushing hard on it can make the real quality worse.

> **A reward model is trusted only near its training data. Optimized hard (**over-optimization**), the policy finds completions it over-rates, and real quality falls.**

<img src="images/rllm2_overoptimization.png" width=62%>

- schematic, invented curves
- the shape is the one [Gao et al. (2022, ICML 2023)](https://arxiv.org/abs/2210.10760) measured
    - a fixed **gold** reward model stood in for people
    - proxy reward models were trained on its labels
- how a policy over-rates: it drifts to what the proxy likes and people do not
    - for example extra length or a confident tone (our reading)
- the reward the policy is trained on keeps rising
- the quality people would give peaks, then falls

**What limits over-optimization**

- the **KL term** to the reference model keeps the policy near where the reward model was trained
- training for fewer steps, or with a smaller learning rate
- a larger or better reward model, or several of them
- a check: sample completions late in training and read them
- none removes the problem: the reward model stays a proxy (our reading)


# What we established, and what it sets up

- **Established**
    - preference data records comparisons $(\x, \y^+, \y^-)$, often easier to collect than scores
    - Bradley-Terry: the probability of a preference is the sigmoid of a reward difference
    - a reward model is a language model with a scalar head, trained on comparisons
    - RLHF: SFT, a reward model, then RL with a KL penalty to the reference model
    - one reward per completion feeds the same per-token machinery as Part B
- **Sets up**
    - **the limit of RLHF:** a reward model is a proxy, and optimizing it hard over-optimizes
    - human labels are also costly and slow
    - next session, RLLM-3: replace the human labeler by a model (RLAIF, reinforcement learning from AI feedback) and by written principles
    - RLLM-10: DPO (direct preference optimization) learns from the same data without a separate reward model
